# LongFlow — head-v2 training run (dual-stream + σ-bucket)

Runtime: **A100 GPU** (real spend; wall-clock over $/hr — standing call,
2026-08-16). ~2–3 h total. Pre-registered criteria: NOTES.md
"HEAD-V2 BUILT + HV2 RUN PRE-REGISTRATION" (2026-08-17).

What this run is: the first training that actually USES capture v2's
dual-stream (`neg_hidden`) + σ schema. Two arms on the SAME filtered pool
(27 truncated files excluded per `capture_v2_audit_flags.json`):

- **Arm A** `hv2_20k` — head-v2: `dual_stream=True, sigma_buckets=4`.
  Guidance absorbed into the field; no CFG at inference.
- **Arm B** `hv2ctl_20k` — v1 architecture, cond-only (control). Evaluated
  under GN8's operating config (heun8 + inference CFG 1.3).

**HARD CONSTRAINT 6 — the gate cell (3) runs FIRST and you LISTEN before
running cell 4.** Expected at 5K steps: shaky-but-intelligible speech (the
ablation baseline). Silence/collapse/alien noise = STOP, paste findings.

| cell | what |
|---|---|
| 1 | cold start (model + repo + bulk-copy cache to local disk) |
| 2 | filter flagged files, filename-bin held-out split, build pools |
| 3 | **GATE: arm A 5K steps + 4 clips → Drive → LISTEN** |
| 4 | real run: arm A + arm B, 20K steps each, ckpts every 5K |
| 5 | held-out teacher-forced renders per checkpoint per arm |
| 6 | closed loop: GN8 protocol, 2 seeds/arm + the σ=0.2-told arm |
| 7 | bundle → Drive (`headv2_eval.zip`) for the GPU scorer notebook |


In [ ]:
# ===== COLD START (idempotent) — run me first, wait for READY =====
NOTEBOOK_VERSION = "Head-v2 train v1.0 (2026-08-17): dual-stream + sigma-bucket, 2 arms + closed loop"
print(f"*** {NOTEBOOK_VERSION} ***")
%cd /content
!git clone -q https://github.com/vibevoice-community/VibeVoice.git 2>/dev/null || true
%cd /content/VibeVoice
!git checkout -q 07cb79fea
!pip install -q -e .

import torch
from vibevoice.modular.modeling_vibevoice_inference import (
    VibeVoiceForConditionalGenerationInference,
)
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
MODEL_ID = "microsoft/VibeVoice-1.5B"
model = VibeVoiceForConditionalGenerationInference.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda"
)
processor = VibeVoiceProcessor.from_pretrained(MODEL_ID)
model.eval()

from google.colab import drive
drive.mount("/content/drive")
import glob, json, os, shutil, sys, time
from pathlib import Path
import numpy as np
import soundfile as sf

if os.path.exists("/content/LongFlow/src"):
    !cd /content/LongFlow && git pull -q
else:
    !git clone -q https://github.com/Josh-E-S/LongFlow.git /content/LongFlow || true
assert os.path.exists("/content/LongFlow/src"), "clone failed — check repo access"
sys.path.insert(0, "/content/LongFlow")
!cd /content/LongFlow && git log --oneline -1

from src.cache.capture import load_utterance
from src.cache.noise import NoiseIntervention
from src.flow_head.cfm import heun_sample
from src.flow_head.integration import CFGFlowHeadPatch, DualStreamFlowHeadPatch, _CFGField
from src.flow_head.model import FlowHead, FlowHeadConfig, sigma_to_bucket
from src.flow_head.trainer import (
    filter_flagged, load_checkpoint, pairs_from_files, sample_latents,
    save_checkpoint, train, PairData,
)
print("DualStreamFlowHeadPatch import OK — repo has the head-v2 commit")

CACHE_V2_DRIVE = "/content/drive/MyDrive/longflow_p1_cache_v2"
CKPT_DIR = "/content/drive/MyDrive/longflow_p1_ckpt"
TRAIN_CACHE_V1 = "/content/drive/MyDrive/longflow_p1_cache"
EVAL_CACHE_DIR = "/content/drive/MyDrive/longflow_p1_evalcache"
GATE3_DIR = "/content/drive/MyDrive/longflow_gate3"
OUT = "/content/headv2"
DRIVE_OUT = "/content/drive/MyDrive/longflow_headv2"
EVAL_DIR = "/content/headv2_eval"
for d in (OUT, DRIVE_OUT, EVAL_DIR):
    os.makedirs(d, exist_ok=True)

# bulk-copy the cache to local disk FIRST — per-file reads off the Drive FUSE
# mount are the trap hit twice on 2026-08-17 (ablation pool C; scorer glob)
LOCAL_CACHE = "/content/cache_v2"
if not os.path.exists(LOCAL_CACHE) or len(glob.glob(f"{LOCAL_CACHE}/*.pt")) < 240:
    os.makedirs(LOCAL_CACHE, exist_ok=True)
    print("bulk-copying cache v2 to local disk (~4 GB, a few minutes)...", flush=True)
    !cp {CACHE_V2_DRIVE}/*.pt {LOCAL_CACHE}/
print(f"{len(glob.glob(f'{LOCAL_CACHE}/*.pt'))} cache files local")

if os.path.exists(f"{DRIVE_OUT}/headv2_report.json"):
    with open(f"{DRIVE_OUT}/headv2_report.json") as f:
        report = json.load(f)
    print(f"resuming: {len(report['runs'])} runs already recorded")
else:
    report = {"notebook_version": NOTEBOOK_VERSION, "runs": []}

def done(tag):
    return os.path.exists(f"{DRIVE_OUT}/{tag}.wav")

def save_wav(tag, wav, meta):
    sf.write(f"{OUT}/{tag}.wav", wav, 24000)
    shutil.copy(f"{OUT}/{tag}.wav", f"{DRIVE_OUT}/{tag}.wav")  # Drive FIRST, every wav (GN3 lesson)
    report["runs"] = [r for r in report["runs"] if r.get("tag") != tag] + [{"tag": tag, **meta}]
    with open(f"{DRIVE_OUT}/headv2_report.json", "w") as f:
        json.dump(report, f, indent=2)
    print(f"saved {tag}: {len(wav)/24000:.1f}s  {meta}", flush=True)

def gen_inputs(texts, prompt_lists):
    inputs = processor(text=texts, voice_samples=prompt_lists,
                       return_tensors="pt", padding=True)
    return {k: (v.to("cuda") if hasattr(v, "to") else v) for k, v in inputs.items()}

def decode_latents(z, chunk_frames=225):  # 225 frames = 30s @ 7.5Hz (gate_v2 OOM fix)
    sc = model.model.speech_scaling_factor
    bi = model.model.speech_bias_factor
    z = z.to("cuda", torch.bfloat16)
    z = z / sc - bi
    wavs, shape_fn = [], None
    for i in range(0, z.shape[0], chunk_frames):
        chunk = z[i : i + chunk_frames]
        candidates = [shape_fn] if shape_fn is not None else [
            lambda c: c.unsqueeze(0), lambda c: c.unsqueeze(0).transpose(1, 2)
        ]
        decoded = None
        for fn in candidates:
            try:
                out = model.model.acoustic_tokenizer.decode(fn(chunk))
                decoded = out[0] if isinstance(out, tuple) else out
                shape_fn = fn
                break
            except Exception as e:
                print(f"decode attempt {tuple(fn(chunk).shape)} failed: {repr(e)[:150]}")
        if decoded is None:
            raise RuntimeError("both decode shapes failed — paste the errors to Claude")
        wavs.append(decoded.detach().float().cpu().numpy().squeeze())
        del out, decoded
        torch.cuda.empty_cache()
    return np.concatenate(wavs) if len(wavs) > 1 else wavs[0]

print("READY")


In [ ]:
# ===== Filter flagged files + FILENAME-bin held-out split + pools =====
HELD_OUT_PER_BIN = 5
FLAGS = "/content/LongFlow/experiments/p1_flow_head/capture_v2_audit_flags.json"

all_files = sorted(glob.glob(f"{LOCAL_CACHE}/*.pt"))
clean_files = filter_flagged(all_files, FLAGS)

def fname_bin(path):  # cv2_1200w_4f19765a.pt -> 1200; NEVER meta["target_words"] (corrupted)
    return int(Path(path).stem.split("_")[1].rstrip("w"))

by_bin = {}
for f in clean_files:
    by_bin.setdefault(fname_bin(f), []).append(f)
held_out_by_bin = {b: fs[:HELD_OUT_PER_BIN] for b, fs in sorted(by_bin.items())}
held_out_files = [f for fs in held_out_by_bin.values() for f in fs]
train_files = [f for b, fs in sorted(by_bin.items()) for f in fs[HELD_OUT_PER_BIN:]]
print({b: len(fs) for b, fs in sorted(by_bin.items())})
print(f"held-out {len(held_out_files)} / train {len(train_files)} scripts")
with open(f"{DRIVE_OUT}/hv2_held_out_manifest.json", "w") as f:
    json.dump({"held_out": [Path(p).name for p in held_out_files],
               "flags_applied": True}, f, indent=2)

t0 = time.time()
data_A = pairs_from_files(train_files, dual_stream=True)   # refuses files missing neg/sigma
data_B = PairData(hidden=data_A.hidden, latent=data_A.latent,
                  mean=data_A.mean, std=data_A.std)         # identical pairs, no v2 fields
print(f"pool: {data_A.hidden.shape[0]} frames  d_model={data_A.d_model}  "
      f"d_latent={data_A.d_latent}  sigma buckets: {torch.bincount(data_A.sigma_bucket).tolist()}  "
      f"({time.time()-t0:.0f}s)")


## 3. GATE — hard constraint 6 (LISTEN before cell 4)

5K steps of arm A, then 4 clips to `Drive/longflow_headv2/gate/`:
one short-bin + one long-bin held-out script, teacher roundtrip + head-v2
heun8. **Expected: intelligible content with the known shaky/underwater 5K
texture (ablation baseline). PASS = content present in both head clips.
Silence, collapse, or alien noise = STOP — paste what you hear.**


In [ ]:
GATE_DIR = f"{DRIVE_OUT}/gate"
os.makedirs(GATE_DIR, exist_ok=True)
if glob.glob(f"{GATE_DIR}/*_hv2gate.wav"):
    print("gate clips already on Drive — skip to listening / cell 4")
else:
    gate_head = FlowHead(FlowHeadConfig(
        d_model=data_A.d_model, d_latent=data_A.d_latent,
        dual_stream=True, sigma_buckets=4))
    print(f"gate head: {gate_head.param_count()/1e6:.2f}M params")
    out = train(gate_head, data_A, steps=5000, batch_size=1024, lr=2e-4,
                ema_decay=0.999, device="cuda", log_every=1000)  # fast EMA for a 5K horizon
    out["ema"].copy_to(gate_head)
    gate_utts = [held_out_by_bin[150][0], held_out_by_bin[1200][0]]
    for fpath in gate_utts:
        utt = load_utterance(fpath)
        wav_t = decode_latents(utt.latent.float())
        sf.write(f"{GATE_DIR}/{utt.utt_id}_teacher.wav", wav_t, 24000)
        z = sample_latents(
            gate_head, utt.hidden.float(), data_A.mean, data_A.std,
            nfe=8, sampler=heun_sample, seed=0,
            neg_condition=utt.neg_hidden.float(),
            sigma_bucket=sigma_to_bucket(utt.sigma.float()).cuda(),
        )
        sf.write(f"{GATE_DIR}/{utt.utt_id}_hv2gate.wav", decode_latents(z), 24000)
        print(f"gate clips for {utt.utt_id} ({fname_bin(fpath)}w) on Drive", flush=True)
    del gate_head
    torch.cuda.empty_cache()
    print("\nLISTEN NOW (Drive/longflow_headv2/gate/) — run cell 4 only on a PASS.")


## 4. The real run — both arms, 20K steps, checkpoints every 5K

Proven recipe (July + train20k conventions): bs 1024, lr 2e-4→2e-5 cosine,
EMA 0.9999, fp32. Checkpoints land on Drive as `hv2_20k_step{N}.pt` /
`hv2ctl_20k_step{N}.pt`. 20K was the established sweet spot at this data
scale (E3: 80K overfits); the 5K-interval checkpoints re-verify that here.


In [ ]:
ARMS = [
    ("hv2_20k", dict(dual_stream=True, sigma_buckets=4), data_A),
    ("hv2ctl_20k", dict(dual_stream=False, sigma_buckets=0), data_B),
]
for tag, cfg_kw, data in ARMS:
    final = f"{CKPT_DIR}/{tag}_step20000.pt"
    if os.path.exists(final):
        print(f"{tag}: final checkpoint already on Drive — skipping")
        continue
    head = FlowHead(FlowHeadConfig(d_model=data.d_model, d_latent=data.d_latent, **cfg_kw))
    print(f"=== {tag}: {head.param_count()/1e6:.2f}M params ===")
    t0 = time.time()
    train(head, data, steps=20000, batch_size=1024, lr=2e-4, lr_final=2e-5,
          ema_decay=0.9999, device="cuda", log_every=1000,
          checkpoint_every=5000,
          checkpoint_path_fn=lambda s, tag=tag: f"{CKPT_DIR}/{tag}_step{s}.pt")
    print(f"{tag} done in {(time.time()-t0)/60:.1f} min")
    del head
    torch.cuda.empty_cache()


## 5. Held-out teacher-forced renders — per checkpoint, per arm

Arm A: native heun8, neg + true per-frame σ buckets (in-distribution eval).
Arm B: heun8 over the CFG-combined field at 1.3 (GN8 operating config) using
the cached neg stream. Full 25 utterances at step 20000; 2-per-bin subset at
intermediate steps (July E3 overfit-curve methodology). seed 0 throughout.


In [ ]:
SUBSET_PER_BIN = 2
FULL_EVAL_STEPS = {20000}
manifest = {"held_out_per_bin": HELD_OUT_PER_BIN, "teacher": {}, "checkpoints": {}}

for fpath in held_out_files:
    utt = load_utterance(fpath)
    name = f"{utt.utt_id}_teacher.wav"
    if not os.path.exists(f"{EVAL_DIR}/{name}"):
        sf.write(f"{EVAL_DIR}/{name}", decode_latents(utt.latent.float()), 24000)
    manifest["teacher"][utt.utt_id] = {"audio": name, "text": utt.text,
                                       "target_words": fname_bin(fpath)}
print(f"{len(manifest['teacher'])} teacher references rendered")

subset_files = [fs[:SUBSET_PER_BIN] for fs in held_out_by_bin.values()]
subset_files = [f for fs in subset_files for f in fs]

def render_arm(arm, head, mean, std, utt):
    if arm == "A":
        z = sample_latents(head, utt.hidden.float(), mean, std, nfe=8,
                           sampler=heun_sample, seed=0,
                           neg_condition=utt.neg_hidden.float(),
                           sigma_bucket=sigma_to_bucket(utt.sigma.float()).cuda())
    else:  # GN8 operating config: heun8 over the CFG-combined field at 1.3
        field = _CFGField(head, utt.neg_hidden.float().cuda(), 1.3)
        g = torch.Generator(device="cuda").manual_seed(0)
        z = heun_sample(field, utt.hidden.float().cuda(), head.cfg.d_latent,
                        nfe=8, sway=0.0, generator=g)
        z = z * std.cuda() + mean.cuda()
    return decode_latents(z)

for step in (5000, 10000, 15000, 20000):
    for arm, tag in (("A", "hv2_20k"), ("B", "hv2ctl_20k")):
        key = f"{step}:{arm}"
        if key in manifest["checkpoints"]:
            continue
        head, mean, std = load_checkpoint(f"{CKPT_DIR}/{tag}_step{step}.pt")
        head = head.to("cuda")
        eval_files = held_out_files if step in FULL_EVAL_STEPS else subset_files
        entries = []
        for fpath in eval_files:
            utt = load_utterance(fpath)
            name = f"{utt.utt_id}_step{step}_{arm}.wav"
            if not os.path.exists(f"{EVAL_DIR}/{name}"):
                sf.write(f"{EVAL_DIR}/{name}", render_arm(arm, head, mean, std, utt), 24000)
            entries.append({"utt_id": utt.utt_id, "audio": name,
                            "teacher_audio": manifest["teacher"][utt.utt_id]["audio"],
                            "text": utt.text, "target_words": fname_bin(fpath), "arm": arm})
        manifest["checkpoints"][key] = entries
        del head
        torch.cuda.empty_cache()
        print(f"step {step} arm {arm}: {len(entries)} renders", flush=True)

with open(f"{EVAL_DIR}/manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
with open(f"{DRIVE_OUT}/manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
print("held-out eval rendered")


## 6. Closed loop — GN8 protocol, the real gate

Same ~800-word turn-split script construction as GN5–GN8 (byte-identical
recipe: v1-cache texts, prompt P0), heun8, `max_new_tokens=3000`. Arms:

- `hv2_heun8_s0/s1` — head-v2 native (DualStreamFlowHeadPatch, σ-bucket 0)
- `hv2ctl_cfg_heun8_s0/s1` — control head + CFGFlowHeadPatch (GN8 config;
  its GN8 July-head twin scored WER 0.031 / voice 61.7 / sim_med 0.522)
- `hv2_heun8_s0_sig02` — informational: σ=0.2 feedback noise applied AND
  told to the head (bucket 2). First test of GameNGen-style σ-conditioning
  at inference; GN6's blind σ=0.2 (g6_sig020) had voice 0%.


In [ ]:
# script construction — identical to GN8 cell 2 (comparability)
def drive_glob(pattern, tries=4, wait=15):
    for i in range(tries):
        hits = sorted(glob.glob(pattern))
        if hits:
            return hits
        print(f"empty listing for {pattern} — retry {i+1}/{tries} in {wait}s", flush=True)
        time.sleep(wait)
    raise RuntimeError(f"still empty after {tries} tries: {pattern}")

V1_LOCAL = "/content/v1texts"
if len(glob.glob(f"{V1_LOCAL}/*.pt")) < 300:
    os.makedirs(V1_LOCAL, exist_ok=True)
    for f in drive_glob(f"{TRAIN_CACHE_V1}/*.pt")[-300:]:  # copy just these 300, then read locally
        shutil.copy(f, V1_LOCAL)
sents = []
for f in sorted(glob.glob(f"{V1_LOCAL}/*.pt")):
    d = torch.load(f, weights_only=True)
    sents.append(d["text"].strip().rstrip(".") + ".")
pool = sents[:200]
P0 = drive_glob(f"{EVAL_CACHE_DIR}/*_prompt.wav")[0]

def turnscript(sentences, target=60, speaker=1):
    turns, cur, w = [], [], 0
    for s in sentences:
        cur.append(s); w += len(s.split())
        if w >= target:
            turns.append(f"Speaker {speaker}: " + " ".join(cur)); cur, w = [], 0
    if cur:
        turns.append(f"Speaker {speaker}: " + " ".join(cur))
    return "\n".join(turns) + "\n"

ABL_WORDS, w = [], 0
for s in pool:
    ABL_WORDS.append(s); w += len(s.split())
    if w >= 800:
        break
ABL_SCRIPT = turnscript(ABL_WORDS)
print("closed-loop script:", w, "words")
report["cl_script"] = ABL_SCRIPT
report["cl_words"] = w

headA, meanA, stdA = load_checkpoint(f"{CKPT_DIR}/hv2_20k_step20000.pt")
headB, meanB, stdB = load_checkpoint(f"{CKPT_DIR}/hv2ctl_20k_step20000.pt")
headA, headB = headA.to("cuda"), headB.to("cuda")

CL_ARMS = [  # (tag, arm, seed, sigma_inf)
    ("hv2_heun8_s0", "A", 0, 0.0),
    ("hv2_heun8_s1", "A", 1, 0.0),
    ("hv2ctl_cfg_heun8_s0", "B", 0, 0.0),
    ("hv2ctl_cfg_heun8_s1", "B", 1, 0.0),
    ("hv2_heun8_s0_sig02", "A", 0, 0.2),
]
for tag, arm, seed, sig in CL_ARMS:
    if done(tag):
        print(f"{tag}: already on Drive — skipping")
        continue
    torch.manual_seed(seed)
    if arm == "A":
        bucket = int(sigma_to_bucket(torch.tensor([sig]))[0]) if sig > 0 else 0
        patch_cm = DualStreamFlowHeadPatch(model, headA, meanA, stdA, nfe=8,
                                           sway=0.0, sampler=heun_sample,
                                           sigma_bucket=bucket)
    else:
        patch_cm = CFGFlowHeadPatch(model, headB, meanB, stdB, nfe=8,
                                    sway=0.0, sampler=heun_sample)
    with patch_cm as patch:
        noise_cm = (
            NoiseIntervention(model.model.acoustic_connector,
                              sigma_fn=lambda c, s=sig: s,
                              active_fn=lambda p=patch: p.calls > 0)
            if sig > 0 else None
        )
        try:
            if noise_cm is not None:
                noise_cm.__enter__()
            with torch.inference_mode():
                gen = model.generate(**gen_inputs([ABL_SCRIPT], [[P0]]),
                                     tokenizer=processor.tokenizer,
                                     cfg_scale=1.3, max_new_tokens=3000)
        finally:
            if noise_cm is not None:
                noise_cm.__exit__(None, None, None)
    wav = gen.speech_outputs[0].detach().float().cpu().numpy().squeeze()
    zs = torch.cat(patch.latents) if patch.latents else torch.zeros(1)
    save_wav(tag, wav, {"arm": arm, "seed": seed, "sigma_inf": sig,
                        "frames": patch.calls,
                        "latent_std": round(float(zs.std()), 3)})


## 7. Bundle → Drive root (the scorer notebook reads it from there)

Includes the GN3 teacher reference so the scorer is self-contained.
Score with `score_headv2_gpu_colab.ipynb` (any GPU runtime — no VibeVoice).


In [ ]:
import zipfile
teacher_ref = f"{GATE3_DIR}/t1_turnsplit_p0.wav"
assert os.path.exists(teacher_ref), "GN3 teacher reference missing from Drive"
shutil.copy(teacher_ref, f"{EVAL_DIR}/t1_turnsplit_p0.wav")
with open(f"{EVAL_DIR}/headv2_report.json", "w") as f:
    json.dump(report, f, indent=2)

ZIP = "/content/drive/MyDrive/headv2_eval.zip"  # Drive ROOT — scorer globs root only
with zipfile.ZipFile(ZIP, "w") as z:
    for f in os.listdir(EVAL_DIR):
        z.write(f"{EVAL_DIR}/{f}", f)
    for f in os.listdir(DRIVE_OUT):
        if f.endswith(".wav"):
            z.write(f"{DRIVE_OUT}/{f}", f"closed_loop/{f}")
print(f"bundle at {ZIP} ({os.path.getsize(ZIP)/1e9:.2f} GB) — run score_headv2_gpu_colab.ipynb next")
